# SKRAID — Analysis Pipeline
**Adithya Ram S | IIIT Sri City | B.Tech Honours ECE**

Skid Risk Assessment for Indian roads. This is the **cell outline** for the full pipeline.
The runnable notebook with code is [`honours_fixed.ipynb`](honours_fixed.ipynb) (13 cells, 0–12).

Run cells in order: 0 → 1 → 2 → … → 12

Dataset (SRIAD) on Drive: https://drive.google.com/drive/folders/1Eg5fG3X_Rem8uqQID1Pa-6Fy4AsNK00O

## Cell 0 — Mount Google Drive

## Cell 1 — List Sensor Files
Quick explorer that lists everything in `sensor_data/`.

## Cell 2 — AVI → MP4 Compression
Compresses raw MJPEG AVI files to H.264 MP4 for Colab processing.

## Cell 3 — Column Normalizer
`normalize_columns()` maps the old `D1–D9` format to the new
`AX/AY/AZ/ROLL/PITCH/YAW/LAT/LON/SPEED_KMH` schema.

## Cell 4 — GPS Frame Extractor
Reads the fusion/sensor/separate CSV, seeks each `FRAME_ID` in the MP4 and
saves GPS-tagged JPEG frames. Handles all three CSV generations.

## Cell 5 — Heatmap Window Frame Extractor
One frame per risk window for the heatmap popups.

**Checkpoint fix:** skip when `len(existing) > 0` (not `len(group)*0.9`) —
window counts shift on every R_skid re-run, but extracted frames don't need redoing.

## Cell 6 — Fusion CSV Explorer + GPS Folium Map
Per-session summary + IMU/GPS/CAM plots, then a speed-colour-coded Folium map.

## Cell 7 — pip install
```
opencv-python pandas numpy tqdm folium
```

## Cell 8 — Vision Feature Extraction
Per-frame: grey-world white balance → WETNESS_RATIO, EDGE_DENSITY_MEAN,
TEXTURE_ROUGHNESS, BRIGHTNESS_VARIANCE, MUD_SCORE.

- **White balance** applied before all HSV thresholds (removes colour cast).
- **mud_score** threshold `s<220` (old `s<150` missed real mud at sat 150–200), saved as an explicit column.
- **Per-session ROI** crop (FULL / 80% / 60%); skips `20260402_164557`.

## Cell 9 — IMU Feature Extraction
Per 2s window: ACCEL_Z_VAR, ROLL_RATE_MEAN, YAW_RATE_MEAN, and SPEED_KMH.

**Speed is derived via the Haversine formula** from consecutive LAT/LON fixes
(`compute_gps_speeds()`) because the recorded sessions have no reliable stored speed.

## Cell 10 — Merger
Merge vision windows + IMU windows → `label_helpers` CSV.

**Critical fix:** `recompute_wetness()` is applied inside `aggregate_vision_to_windows()`
**before** aggregating (tightened HSV thresholds), and `brightness_variance` + `mud_score`
are passed through when present.

## Cell 11 — R_skid Computation + GPS Heatmap
```
R_skid_base  = 0.5*V_vision + 0.3*sigma2(az) + 0.2*(delta_roll + delta_yaw)
R_skid_final = R_skid_base            # speed multiplier currently DISABLED
```
**Vision term = `max(wet, rough, mud)`** — three INDEPENDENT hazard sub-scores, because road
risk is U-shaped in texture (both very smooth/wet AND very rough/gravel are dangerous). Each
is percentile-scaled to [0,1] and reads ~0 on a normal road. `WETNESS_RATIO` (HSV) is NOT used
— it correlates -0.79 with real wet roads (brightness-driven, inverted).
Normalization: vision = pre-scaled 0-1 hazard composite, **IMU/gyro = per-session** min-max.
No-IMU windows get `R_skid_base = V_vision` directly. Motion gate + speed multiplier are
commented out until Haversine speed is confirmed.

## Cell 12 — Analysis Report + Figures